1.Load and validate the dataset

In [17]:

import pandas as pd

df = pd.read_csv("heart_failure_clinical_records_dataset.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nMortality outcome distribution:")
print(df["DEATH_EVENT"].value_counts().sort_index())

print("\nMortality outcome percentages:")
print(
    (df["DEATH_EVENT"].value_counts(normalize=True).sort_index() * 100)
    .round(2)
)


Dataset shape: (299, 13)

Column names:
['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time', 'DEATH_EVENT']

Missing values:
age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0
dtype: int64

Duplicate rows: 0

Mortality outcome distribution:
DEATH_EVENT
0    203
1     96
Name: count, dtype: int64

Mortality outcome percentages:
DEATH_EVENT
0    67.89
1    32.11
Name: proportion, dtype: float64


2.Split the data

In [18]:

from sklearn.model_selection import train_test_split


y = df["DEATH_EVENT"]
# Exclude the target and follow-up time
X = df.drop(columns=["DEATH_EVENT", "time"])


X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20, random_state=42,stratify=y)

# Define the two feature sets
selected_features = ["serum_creatinine", "ejection_fraction"]

X_train_two = X_train[selected_features]
X_test_two = X_test[selected_features]

# Display the split information
print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining outcome distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting outcome distribution:")
print(y_test.value_counts().sort_index())

print("\nModel A training shape:", X_train.shape)
print("Model A testing shape:", X_test.shape)

print("\nModel B training shape:", X_train_two.shape)
print("Model B testing shape:", X_test_two.shape)



Training records: 239
Testing records: 60

Training outcome distribution:
DEATH_EVENT
0    162
1     77
Name: count, dtype: int64

Testing outcome distribution:
DEATH_EVENT
0    41
1    19
Name: count, dtype: int64

Model A training shape: (239, 11)
Model A testing shape: (60, 11)

Model B training shape: (239, 2)
Model B testing shape: (60, 2)


3.Train both baseline models

In [19]:

from sklearn.ensemble import RandomForestClassifier

# Model A: use all 11 features
model_a = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
model_a.fit(X_train, y_train)

# Model B: use only the two selected features
model_b = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
model_b.fit(X_train_two, y_train)

print("\nBoth Random Forest models have been trained successfully.")



Both Random Forest models have been trained successfully.


4.Evaluate both models

In [20]:

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    average_precision_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
from pathlib import Path


# Create a folder for the output figures
output_folder = Path("results")
output_folder.mkdir(exist_ok=True)


def evaluate_model(model, X_test_data, y_test_data, model_name, filename):
    # Predict the outcome using unseen test data
    predictions = model.predict(X_test_data)

    # Probability of death (DEATH_EVENT = 1)
    probabilities = model.predict_proba(X_test_data)[:, 1]

    # Confusion matrix: [[TN, FP], [FN, TP]]
    tn, fp, fn, tp = confusion_matrix(
        y_test_data,
        predictions,
        labels=[0, 1]
    ).ravel()

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test_data, predictions)
    sensitivity = recall_score(
        y_test_data, predictions, zero_division=0
    )
    specificity = tn / (tn + fp) if (tn + fp) else 0
    precision = precision_score(
        y_test_data, predictions, zero_division=0
    )
    f1 = f1_score(
        y_test_data, predictions, zero_division=0
    )
    mcc = matthews_corrcoef(y_test_data, predictions)
    roc_auc = roc_auc_score(y_test_data, probabilities)

    # PR-AUC summary using average precision
    pr_auc = average_precision_score(
        y_test_data, probabilities
    )

    # Display the results
    print(f"\n{'=' * 45}")
    print(model_name)
    print(f"{'=' * 45}")

    print("\nConfusion matrix:")
    print([[tn, fp], [fn, tp]])

    print(f"True negatives (TN): {tn}")
    print(f"False positives (FP): {fp}")
    print(f"False negatives (FN): {fn}")
    print(f"True positives (TP): {tp}")

    print(f"\nAccuracy:     {accuracy:.3f}")
    print(f"Sensitivity:  {sensitivity:.3f}")
    print(f"Specificity:  {specificity:.3f}")
    print(f"Precision:    {precision:.3f}")
    print(f"F1-score:     {f1:.3f}")
    print(f"MCC:          {mcc:.3f}")
    print(f"ROC-AUC:      {roc_auc:.3f}")
    print(f"PR-AUC:       {pr_auc:.3f}")

    # Create and save the confusion matrix figure
    display = ConfusionMatrixDisplay.from_predictions(
        y_test_data,
        predictions,
        labels=[0, 1],
        display_labels=["Survived (0)", "Died (1)"],
        cmap="Blues",
        values_format="d"
    )

    display.ax_.set_title(model_name)
    display.figure_.tight_layout()

    figure_path = output_folder / filename
    display.figure_.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(display.figure_)
    print(f"Confusion matrix saved to: {figure_path}")

    # Return all metrics for the comparison CSV
    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "F1-score": f1,
        "MCC": mcc,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc
    }


# Evaluate both models on the same test patients
results_a = evaluate_model(
    model_a,
    X_test,
    y_test,
    "Model A: Random Forest with 11 Features",
    "figure3_confusion_matrix_model_a.png"
)

results_b = evaluate_model(
    model_b,
    X_test_two,
    y_test,
    "Model B: Random Forest with Two Features",
    "figure4_confusion_matrix_model_b.png"
)


# Figure 5: ROC Curve Comparison

# Get predicted probabilities for both models
probabilities_a = model_a.predict_proba(X_test)[:, 1]
probabilities_b = model_b.predict_proba(X_test_two)[:, 1]

# Calculate ROC curve points
fpr_a, tpr_a, _ = roc_curve(y_test, probabilities_a)
fpr_b, tpr_b, _ = roc_curve(y_test, probabilities_b)

# Draw both ROC curves
plt.figure(figsize=(8, 6))

plt.plot(
    fpr_a, tpr_a,
    label=f"All 11 features (AUC = {roc_auc_score(y_test, probabilities_a):.3f})"
)

plt.plot(
    fpr_b, tpr_b,
    label=f"Two features (AUC = {roc_auc_score(y_test, probabilities_b):.3f})"
)

# Random-classifier reference line
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Sensitivity)")
plt.title("ROC Curve Comparison: Random Forest Models")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()

roc_path = output_folder / "figure5_roc_curve_comparison.png"
plt.savefig(roc_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"ROC curve saved to: {roc_path}")


# Figure 6: Precision-Recall Curve Comparison

# Calculate Precision-Recall curve points
precision_a, recall_a, _ = precision_recall_curve(y_test, probabilities_a)
precision_b, recall_b, _ = precision_recall_curve(y_test, probabilities_b)

# Calculate Average Precision (AP) for each model
ap_a = average_precision_score(y_test, probabilities_a)
ap_b = average_precision_score(y_test, probabilities_b)

# Draw both Precision-Recall curves
plt.figure(figsize=(8, 6))

plt.plot(
    recall_a, precision_a,
    label=f"All 11 features (AP = {ap_a:.3f})"
)

plt.plot(
    recall_b, precision_b,
    label=f"Two features (AP = {ap_b:.3f})"
)

# Reference line: proportion of positive cases in the test set
baseline_precision = y_test.mean()
plt.axhline(
    y=baseline_precision,
    linestyle="--",
    label=f"Positive-class prevalence = {baseline_precision:.3f}"
)

plt.xlabel("Recall (Sensitivity)")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve Comparison: Random Forest Models")
plt.legend(loc="best")
plt.grid(alpha=0.3)
plt.tight_layout()

pr_path = output_folder / "figure6_precision_recall_curve_comparison.png"
plt.savefig(pr_path, dpi=300, bbox_inches="tight")
plt.close()

print(f"Precision-Recall curve saved to: {pr_path}")



Model A: Random Forest with 11 Features

Confusion matrix:
[[np.int64(36), np.int64(5)], [np.int64(13), np.int64(6)]]
True negatives (TN): 36
False positives (FP): 5
False negatives (FN): 13
True positives (TP): 6

Accuracy:     0.700
Sensitivity:  0.316
Specificity:  0.878
Precision:    0.545
F1-score:     0.400
MCC:          0.233
ROC-AUC:      0.800
PR-AUC:       0.581
Confusion matrix saved to: results\figure3_confusion_matrix_model_a.png

Model B: Random Forest with Two Features

Confusion matrix:
[[np.int64(35), np.int64(6)], [np.int64(11), np.int64(8)]]
True negatives (TN): 35
False positives (FP): 6
False negatives (FN): 11
True positives (TP): 8

Accuracy:     0.717
Sensitivity:  0.421
Specificity:  0.854
Precision:    0.571
F1-score:     0.485
MCC:          0.302
ROC-AUC:      0.731
PR-AUC:       0.650
Confusion matrix saved to: results\figure4_confusion_matrix_model_b.png
ROC curve saved to: results\figure5_roc_curve_comparison.png
Precision-Recall curve saved to: results\f

5.comparison table

In [21]:

results_df = pd.DataFrame([results_a, results_b])

print("\n\nFINAL MODEL COMPARISON")
print(results_df.round(3).to_string(index=False))

# Save the results for your coursework
results_df.to_csv("baseline_model_comparison.csv", index=False)

print("\nResults saved to baseline_model_comparison.csv")




FINAL MODEL COMPARISON
                                   Model  Accuracy  Sensitivity  Specificity  Precision  F1-score   MCC  ROC-AUC  PR-AUC
 Model A: Random Forest with 11 Features     0.700        0.316        0.878      0.545     0.400 0.233    0.800   0.581
Model B: Random Forest with Two Features     0.717        0.421        0.854      0.571     0.485 0.302    0.731   0.650

Results saved to baseline_model_comparison.csv


6.Comparison with the Original Research Paper

In [22]:

# Published Random Forest results from Chicco and Jurman (2020)
paper_results = pd.DataFrame([
    {
        "Model": "All features",
        "Accuracy": 0.740,
        "Sensitivity": 0.491,
        "Specificity": 0.864,
        "F1-score": 0.547,
        "MCC": 0.384,
        "ROC-AUC": 0.800
    },
    {
        "Model": "Two features",
        "Accuracy": 0.585,
        "Sensitivity": 0.541,
        "Specificity": 0.855,
        "F1-score": 0.754,
        "MCC": 0.418,
        "ROC-AUC": 0.698
    }
])

# Keep only the corresponding metrics from your own results
our_results = results_df[
    ["Model", "Accuracy", "Sensitivity", "Specificity",
     "F1-score", "MCC", "ROC-AUC"]
].copy()

# Give the models clearer names for comparison
our_results["Model"] = our_results["Model"].replace({
    "Model A: Random Forest with 11 Features": "All features",
    "Model B: Random Forest with Two Features": "Two features",
    "Model A: All 11 features": "All features",
    "Model B: Two selected features": "Two features"
})

# Combine results into one table
comparison_table = pd.concat([
    paper_results.assign(Source="Original paper"),
    our_results.assign(Source="my implementation")
], ignore_index=True)

print("\nCOMPARISON WITH THE ORIGINAL PAPER")
print(comparison_table.round(3).to_string(index=False))

comparison_table.to_csv(
    "paper_vs_baseline_comparison.csv",
    index=False
)

print("\nComparison saved to paper_vs_baseline_comparison.csv")



COMPARISON WITH THE ORIGINAL PAPER
       Model  Accuracy  Sensitivity  Specificity  F1-score   MCC  ROC-AUC            Source
All features     0.740        0.491        0.864     0.547 0.384    0.800    Original paper
Two features     0.585        0.541        0.855     0.754 0.418    0.698    Original paper
All features     0.700        0.316        0.878     0.400 0.233    0.800 my implementation
Two features     0.717        0.421        0.854     0.485 0.302    0.731 my implementation

Comparison saved to paper_vs_baseline_comparison.csv


### Comparison to the Original Study

The results obtained with my baseline implementation were compared with those reported in Chicco and Jurman (2020). I got an accuracy of 70.0% for the model with all 11 features, whereas the original paper got 74.0%. Both models achieved a ROC-AUC of roughly 0.800. However, my model was less sensitive, with a lower F1-score and MCC.

My Random Forest model with only serum creatinine and ejection fraction got 71.7% accuracy compared to 58.5% in the original paper. My 2-feature model had higher sensitivity, F1-score and MCC than my 11-feature model. Its average precision (PR-AUC) was 0.650 versus 0.581 for the all-feature model. But the two feature model had lower ROC-AUC and specificity.

The two features model was superior to the other model in terms of accuracy, sensitivity, precision, F1-score, MCC and Average Precision in my experiment. The all-feature model performed better in specificity and ROC-AUC. This indicates that the combination of serum creatinine and ejection fraction can provide useful predictive information, but does not prove that a better model will always be obtained with only two features.

This comparison is not without bounds. The original study reported average performance over 100 runs, while I used a single stratified train/test split with 60 patients in the test set. Thus, differences may be due to the data split, the model configuration, and the evaluation procedure. Repeated stratified evaluation and validation on an independent dataset would be helpful in establishing the robustness of these performance patterns.